# 06 Reporting and Export

Этот ноутбук создаёт **финальные отчёты и экспортирует результаты**:

1. Сводная таблица метрик по всем моделям
2. Предсказания (genomic predictions)
3. Экспорт в форматы для дальнейшего анализа (CSV, Excel)

**Для кого**: для тех, кто хочет сохранить результаты и использовать их дальше — в других программах, для визуализации в Tableau/PowerBI, или для передачи коллегам.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from pathlib import Path

from gp_py.io import fn_load_genotype, fn_load_phenotype, fn_filter_genotype, fn_filter_phenotype, fn_merge_genotype_and_phenotype
from gp_py.cv import fn_cross_validation_within_population
from gp_py.models import fn_ridge, fn_lasso, fn_elastic_net
from gp_py.schema import MergedData

## 1. Запускаем полный пайплайн для примера

In [ ]:
DATA_DIR = Path("../inst/exec_Rscript/input")

G = fn_load_genotype(str(DATA_DIR / "test_geno.Rds"))
list_pheno = fn_load_phenotype(str(DATA_DIR / "test_pheno.tsv"))
Gf = fn_filter_genotype(G)
phf = fn_filter_phenotype(list_pheno)
merged = fn_merge_genotype_and_phenotype(Gf, phf)

# Разделяем на известные и пропущенные
known_mask = merged.y.notna()
missing_mask = merged.y.isna()

merged_known = MergedData(
    G=merged.G.loc[known_mask].copy(),
    y=merged.y.loc[known_mask].copy(),
    pop=merged.pop.loc[known_mask].copy(),
    trait_name=merged.trait_name,
)

# CV
cv_results = fn_cross_validation_within_population(
    merged_known,
    n_folds=3,
    n_reps=2,
    vec_models_to_test=("ridge", "lasso", "elastic_net"),
    bool_parallel=False,
    bayes_backend="native",
    verbose=False
)

print("CV выполнен!")

## 2. Сводная таблица метрик

In [ ]:
metrics_df = cv_results["METRICS_WITHIN_POP"]

# Агрегируем
summary = metrics_df.groupby("model").agg({
    "corr": ["mean", "std", "min", "max"],
    "rmse": ["mean", "std"],
    "mae": ["mean", "std"],
    "r2": ["mean", "std"]
}).round(4)

print("=== Сводка метрик ===")
print(summary.to_string())

## 3. Лучшая модель и предсказания

In [ ]:
# Выбираем лучшую
best_model = metrics_df.groupby("model")["corr"].mean().idxmax()
print(f"Лучшая модель: {best_model}")

# Предсказания
MODEL_MAP = {"ridge": fn_ridge, "lasso": fn_lasso, "elastic_net": fn_elastic_net}
fn_best = MODEL_MAP[best_model]

train_idx = list(range(known_mask.sum()))
missing_idx = list(range(known_mask.sum(), known_mask.sum() + missing_mask.sum()))

result = fn_best(merged, train_idx, missing_idx, other_params={"bayes_backend": "native"}, verbose=False)
predictions = result["df_y_validation"]

print(f"\nПредсказано {len(predictions)} записей")

## 4. Экспорт в файлы

In [ ]:
EXPORT_DIR = Path("../output/export")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Экспорт метрик
metrics_df.to_csv(EXPORT_DIR / "metrics_all_folds.csv", index=False)
summary.to_csv(EXPORT_DIR / "metrics_summary.csv")

# Экспорт предсказаний
predictions.to_csv(EXPORT_DIR / "genomic_predictions.csv", index=False)

print("Экспортировано:")
for f in EXPORT_DIR.iterdir():
    print(f"  {f.name}")

## 5. Пример использования результатов

In [ ]:
# Показать пример — топ предсказаний по каждой популяции
print("=== Топ предсказаний по популяциям ===")
for pop in predictions["pop"].unique():
    subset = predictions[predictions["pop"] == pop].nlargest(3, "y_pred")
    print(f"\n{pop}:")
    print(subset[["id", "y_pred"]].to_string(index=False))

## Итог

- Мы создали сводную таблицу метрик по всем моделям
- Выбрали лучшую модель по CV
- Предсказали пропущенные фенотипы
- Экспортировали результаты в CSV

**Что дальше**:
- Использовать предсказания для селекции (топ особи с высоким y_pred)
- Передать результаты в R для финального сравнения
- Подключить визуализацию (Plotly/Shiny) как в `inst/plot_gs_gp/app.R`